In [18]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch, Rectangle
import ast
import folium
from folium.plugins import MarkerCluster
import reverse_geocoder as rg
import re
import pycountry
import os
import numpy as np
import geopandas as gpd
import fiona
import sys
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import ruptures as rpt
from haversine import haversine
import functions as own
from timezonefinder import TimezoneFinder
import zoneinfo
from scipy.stats import gaussian_kde, chi2_contingency
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from libpysal.weights import lat2W
from libpysal.weights import Queen
from libpysal.weights import DistanceBand
from esda.moran import Moran_Local, Moran
from shapely.geometry import box
from pymannkendall import original_test
from statsmodels.stats.multitest import multipletests
from pyproj import Transformer
import importlib

In [19]:
importlib.reload(own)

<module 'functions' from 'c:\\Studium\\X_Masterarbeit\\Data\\Master_Thesis\\Code\\functions.py'>

country observation numbers and user groups

In [20]:
# load and merge datasets
df_super = pd.read_csv("../CWData_superusers.csv")
df_between = pd.read_csv("../CWData_between.csv")
df_onetimers = pd.read_csv("../CWData_onetimers.csv")

df_super["user_group"] = "Superuser"
df_between["user_group"] = "Betweener"
df_onetimers["user_group"] = "One-timer"

df_all = pd.concat([df_super, df_between, df_onetimers], ignore_index=True)

# no countries with too few obs
country_totals = df_all.groupby("Country").size()
valid_countries = country_totals[country_totals >= 100].index
df_filtered = df_all[df_all["Country"].isin(valid_countries)].copy()

print(f"Countries included: {len(valid_countries)} (out of {df_all['Country'].nunique()})")
print(f"Observations included: {len(df_filtered)} (out of {len(df_all)})")

# contingency table
contingency_table = pd.crosstab(df_filtered["Country"], df_filtered["user_group"])

# chi square
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"\nChi2 = {chi2:.2f}, dof = {dof}, p-value = {p:.4g}")

v = own.cramers_v(chi2, contingency_table)
print(f"Cramér's V = {v:.3f}")

# results
expected_df = pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns)
residuals = (contingency_table - expected_df) / np.sqrt(expected_df)

# most important countries per user group
top_n = 5
pd.set_option("display.max_rows", None)
print(f"\nTop {top_n} countries with more Superuser observations than expected:")
print(residuals["Superuser"].sort_values(ascending=False).head(top_n).round(2))

print(f"\nTop {top_n} countries with more Betweener observations than expected:")
print(residuals["Betweener"].sort_values(ascending=False).head(top_n).round(2))

print(f"\nTop {top_n} countries with more One-timer observations than expected:")
print(residuals["One-timer"].sort_values(ascending=False).head(top_n).round(2))

# export
residuals_export = residuals.round(2)
residuals_export.to_csv("../Products/CSVs/country_user_group_residuals.csv")

contingency_table.to_csv("../Products/CSVs/country_user_group_contingency.csv")

C:\Users\yanni\AppData\Local\Temp\ipykernel_43704\3414622426.py:2: DtypeWarning: Columns (0: SoilMoisture, 1: Plastic_Amount, 2: Plastic_Location, 3: Plastic_RiverWidth, 4: Plastic_PET, 5: Plastic_POSoft, 6: Plastic_POHard, 7: Plastic_PS, 8: Plastic_PSE, 9: Plastic_PMultilayer, 10: Plastic_POther, 11: Plastic_Shore_Plotsize, 12: Plastic_Removed, 13: Waterlevel_Physical_Unit, 14: Streamtype, 15: Swimming_Quality, 16: Drinking_Quality, 17: Naturality, 18: StreamColor, 19: Stream_Ground_Visibility, 20: Stream_Animals, 21: Stream_Pollution_Reason, 22: Stream_sometimes_dry, 23: Stream_Name, 24: TempStream_snow_ice, 25: Stream_Waterquality, 26: Stream_Waterclarity, 27: Stream_Vegetation, 28: Stream_Foam, 29: Stream_Algae, 30: Stream_Odor, 31: Stream_Odor_Type, 32: Stream_Odor_Type_other, 33: Stream_Litter, 34: Stream_typical_Color) have mixed types. Specify dtype option on import or set low_memory=False.
  df_super = pd.read_csv("../CWData_superusers.csv")
C:\Users\yanni\AppData\Local\Temp\i

Countries included: 30 (out of 90)
Observations included: 67543 (out of 68586)

Chi2 = 20951.42, dof = 58, p-value = 0
Cramér's V = 0.394

Top 5 countries with more Superuser observations than expected:
Country
Switzerland       30.60
Canada            26.31
Spain             24.97
Austria           21.19
United Kingdom    11.14
Name: Superuser, dtype: float64

Top 5 countries with more Betweener observations than expected:
Country
France                      28.96
United States of America    25.77
Estonia                     25.35
Italy                       22.74
Ireland                     18.93
Name: Betweener, dtype: float64

Top 5 countries with more One-timer observations than expected:
Country
Philippines    51.57
Costa Rica     20.08
Brazil         16.08
Norway         13.43
Switzerland     8.13
Name: One-timer, dtype: float64


user groups and distributions of time of day, month of year and years

In [21]:
df_super = pd.read_csv("../CWData_superusers.csv", low_memory=False)
df_between = pd.read_csv("../CWData_between.csv", low_memory=False)
df_onetimers = pd.read_csv("../CWData_onetimers.csv", low_memory=False)

df_super["user_group"] = "Superuser"
df_between["user_group"] = "Betweener"
df_onetimers["user_group"] = "One-timer"

df_all = pd.concat([df_super, df_between, df_onetimers], ignore_index=True)
df_all["created_at_local"] = pd.to_datetime(df_all["created_at_local"])

df_all["hour"] = df_all["created_at_local"].dt.hour
df_all["month"] = df_all["created_at_local"].dt.month
df_all["year"] = df_all["created_at_local"].dt.year

def run_chi2_analysis(df, time_col, label, top_n=5):
    contingency_table = pd.crosstab(df[time_col], df["user_group"])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    v = own.cramers_v(chi2, contingency_table)

    expected_df = pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns)
    residuals = (contingency_table - expected_df) / np.sqrt(expected_df)

    print(f"--- {label} ---")
    print(f"Chi2 = {chi2:.2f}, dof = {dof}, p-value = {p:.4g}, Cramér's V = {v:.3f}\n")

    for group in ["Superuser", "Betweener", "One-timer"]:
        print(f"Top {top_n} {time_col} bins with more {group} observations than expected:")
        print(residuals[group].sort_values(ascending=False).head(top_n).round(2))
        print()

    return contingency_table, residuals, chi2, p, v

# analysis for hour, month, year
hour_table, hour_residuals, hour_chi2, hour_p, hour_v = run_chi2_analysis(df_all, "hour", "Hour of Day")
month_table, month_residuals, month_chi2, month_p, month_v = run_chi2_analysis(df_all, "month", "Month of Year")
df_all_full_years = df_all[~df_all["year"].isin([2017, 2026])]
year_table, year_residuals, year_chi2, year_p, year_v = run_chi2_analysis(df_all_full_years, "year", "Year (full years only)")

# export
hour_residuals.round(2).to_csv("../Products/CSVs/hour_user_group_residuals.csv")
month_residuals.round(2).to_csv("../Products/CSVs/month_user_group_residuals.csv")
year_residuals.round(2).to_csv("../Products/CSVs/year_user_group_residuals.csv")

--- Hour of Day ---
Chi2 = 2383.91, dof = 46, p-value = 0, Cramér's V = 0.132

Top 5 hour bins with more Superuser observations than expected:
hour
8     25.74
17     8.21
7      7.51
9      4.16
18     2.65
Name: Superuser, dtype: float64

Top 5 hour bins with more Betweener observations than expected:
hour
11    7.50
6     7.29
10    6.74
14    6.31
0     5.76
Name: Betweener, dtype: float64

Top 5 hour bins with more One-timer observations than expected:
hour
11    6.40
14    4.40
15    4.16
10    3.77
3     2.74
Name: One-timer, dtype: float64

--- Month of Year ---
Chi2 = 795.28, dof = 22, p-value = 5.663e-154, Cramér's V = 0.076

Top 5 month bins with more Superuser observations than expected:
month
12    9.32
1     6.10
7     3.62
11    3.50
10    3.42
Name: Superuser, dtype: float64

Top 5 month bins with more Betweener observations than expected:
month
3    8.14
5    7.56
4    2.58
6    1.28
8    1.12
Name: Betweener, dtype: float64

Top 5 month bins with more One-timer observ

user groups and category distribution (global)

In [23]:
df_super = pd.read_csv("../CWData_superusers.csv", low_memory=False)
df_between = pd.read_csv("../CWData_between.csv", low_memory=False)
df_onetimers = pd.read_csv("../CWData_onetimers.csv", low_memory=False)

df_super["user_group"] = "Superuser"
df_between["user_group"] = "Betweener"
df_onetimers["user_group"] = "One-timer"

df_all = pd.concat([df_super, df_between, df_onetimers], ignore_index=True)
df_all["created_at_local"] = pd.to_datetime(df_all["created_at_local"])

contingency_table = pd.crosstab(df_all["Category"], df_all["user_group"])

chi2, p, dof, expected = chi2_contingency(contingency_table)
v = own.cramers_v(chi2, contingency_table)

expected_df = pd.DataFrame(expected, index=contingency_table.index, columns=contingency_table.columns)
residuals = (contingency_table - expected_df) / np.sqrt(expected_df)

print(f"Chi2 = {chi2:.2f}, dof = {dof}, p-value = {p:.4g}, Cramér's V = {v:.3f}\n")

print("Category shares by user group (%):")
category_shares = pd.crosstab(df_all["Category"], df_all["user_group"], normalize="columns") * 100
print(category_shares.round(1))

print("\nStandardized residuals:")
print(residuals.round(2))

# Export
category_shares.round(1).to_csv("../Products/CSVs/category_by_user_group_shares.csv")
residuals.round(2).to_csv("../Products/CSVs/category_by_user_group_residuals.csv")

Chi2 = 5249.52, dof = 12, p-value = 0, Cramér's V = 0.196

Category shares by user group (%):
user_group           Betweener  One-timer  Superuser
Category                                            
physical scale             6.8        3.5       12.5
plastic pollution          7.1        9.7        0.2
soil moisture              5.4        6.0        4.4
standing water type        1.3        2.6        0.0
stream type                6.3       16.5        3.2
temporary stream          31.8       26.4       47.5
virtual scale             41.4       35.4       32.1

Standardized residuals:
user_group           Betweener  One-timer  Superuser
Category                                            
physical scale          -15.51      -6.93      19.00
plastic pollution        27.73       9.94     -33.43
soil moisture             3.49       1.66      -4.30
standing water type      11.68       7.77     -14.86
stream type               9.75      18.37     -14.95
temporary stream        -20.92   